# 🚀 Mirror-Leech-Telegram-Bot (MLTB) on Google Colab
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aminelazz/mirror-leech-telegram-bot/blob/master/MLTB_Colab.ipynb)

A feature-rich, asynchronous Telegram bot designed to mirror and leech files from Torrents (Aria2 / qBittorrent), Usenet (SABnzbd), YouTube (`yt-dlp`), direct links, and cloud services to **Google Drive**, **Telegram**, or **Rclone**-supported remotes.

---

### 📋 Instructions & Workflow:
1. **Cell 1: Anti-Timeout Keep-Alive** ⏱️ - Keeps your Colab session from disconnecting during idle periods.
2. **Cell 2: Mount Google Drive (Optional)** 💾 - Persists credentials (`config.py`, `token.pickle`, `rclone.conf`, `accounts.zip`) so you never have to re-enter them!
3. **Cell 3: Setup Dependencies & Environment** ⚙️ - Installs system packages (`aria2`, `qbittorrent-nox`, `sabnzbdplus`, `ffmpeg`, `rclone`, etc.), creates directory mappings, and installs Python requirements.
4. **Cell 4: Bot Configuration** 🔑 - Configure your bot tokens and options via the interactive Colab Form.
5. **Cell 5: Upload Private Files (Optional)** 📂 - Upload `token.pickle`, `rclone.conf`, or `accounts.zip` if needed.
6. **Cell 6: Start Bot & View Live Logs** 🚀 - Starts all download engines and runs the bot with live streaming logs.
7. **Cell 7: Emergency Stop / Cleanup** 🛑 - Safely stops the bot and terminates background services.



In [ ]:
# @title ⏱️ Anti-Timeout Keep-Alive { form-width: "40%" }
# @markdown Run this cell to prevent Google Colab from disconnecting during idle periods or background tab suspension.
# @markdown You can also paste this snippet in your browser's Developer Console (F12 -> Console):
# @markdown ```javascript
# @markdown function ClickConnect(){ console.log("Colab ping"); document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect")?.click(); }
# @markdown setInterval(ClickConnect, 60000);
# @markdown ```

from IPython.display import HTML, display

display(HTML('''
    <div style="padding: 10px; background-color: #1e1e1e; border-radius: 8px; color: #4CAF50; font-family: monospace;">
        <b>🔄 Keep-Alive Active:</b> Audio oscillator context and interval ping initialized to prevent browser tab throttling.
    </div>
    <audio id="keepAliveAudio" loop>
        <source src="data:audio/wav;base64,UklGRiQAAABXQVZFZm10IBAAAAABAAEARKwAAIhYAQACABAAZGF0YQAAAAA=" type="audio/wav">
    </audio>
    <script>
        const audio = document.getElementById('keepAliveAudio');
        if (audio) {
            audio.play().catch(e => console.log('Audio keepalive muted'));
        }
        function pingColab() {
            try {
                const btn = document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect");
                if (btn) btn.click();
            } catch (err) {}
        }
        setInterval(pingColab, 60000);
    </script>
'''))
print("✅ Keep-Alive script initialized.")



In [ ]:
# @title 📁 Step 1: Mount Google Drive (Optional) { form-width: "50%" }
# @markdown Mount Google Drive to persist your configuration, tokens (`token.pickle`), rclone remotes (`rclone.conf`), and Service Accounts (`accounts.zip`).
MOUNT_DRIVE = True # @param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/MLTB" # @param {type:"string"}

import os

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        print(f"✅ Google Drive mounted successfully at /content/drive")
        print(f"📁 Persistence directory: {DRIVE_DIR}")
        print("💡 Any files placed in this folder will be auto-detected by MLTB!")
    except Exception as e:
        print(f"⚠️ Could not mount Google Drive: {e}")
else:
    print("⏭️ Skipped mounting Google Drive.")



In [ ]:
# @title 🛠️ Step 2: Install System Packages, Rclone & Setup Repo { form-width: "60%" }
# @markdown Configure repository source and installation options:
REPO_URL = "https://github.com/aminelazz/mirror-leech-telegram-bot.git" # @param {type:"string"}
BRANCH = "master" # @param {type:"string"}
FORCE_REINSTALL = False # @param {type:"boolean"}

import os
import sys
import subprocess
import shutil

def run_cmd(cmd, check=True):
    res = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if check and res.returncode != 0:
        print(f"❌ Command failed: {cmd}\nError: {res.stderr.strip()}")
        raise RuntimeError(res.stderr.strip())
    return res.stdout.strip()

print("📦 [1/4] Installing system dependencies via apt...")
# Update and install aria2, qbittorrent-nox, sabnzbdplus, ffmpeg, p7zip, par2, libmagic, libattr
run_cmd(
    "apt-get update -qq && apt-get install -y -qq --no-install-recommends "
    "aria2 qbittorrent-nox sabnzbdplus python3-sabctools p7zip-full p7zip-rar ffmpeg "
    "libmagic-dev libattr1-dev par2 curl wget git"
)
print("   ✅ System utilities installed (aria2, qbittorrent-nox, sabnzbdplus, ffmpeg, 7z, par2).")

print("📦 [2/4] Checking and installing rclone...")
if not shutil.which("rclone"):
    run_cmd("curl -fsSL https://rclone.org/install.sh | bash")
    print("   ✅ Rclone installed successfully.")
else:
    print("   ✅ Rclone is already installed.")

print("📂 [3/4] Preparing repository & directory symlinks...")
TARGET_DIR = "/content/mirror-leech-telegram-bot"
current_dir = os.getcwd()

if os.path.exists(os.path.join(current_dir, "bot", "__main__.py")):
    TARGET_DIR = current_dir
    print(f"   ℹ️ Already inside repository directory: {TARGET_DIR}")
elif os.path.exists(TARGET_DIR) and not FORCE_REINSTALL:
    print(f"   ℹ️ Repository already cloned at {TARGET_DIR}")
    os.chdir(TARGET_DIR)
else:
    if os.path.exists(TARGET_DIR):
        shutil.rmtree(TARGET_DIR)
    print(f"   📥 Cloning {REPO_URL} (branch: {BRANCH})...")
    run_cmd(f"git clone -b {BRANCH} {REPO_URL} {TARGET_DIR}")
    os.chdir(TARGET_DIR)

# Crucial symlinks: /app points to repo dir, /usr/src/app points to /app
run_cmd(f"ln -sfn {TARGET_DIR} /app")
run_cmd("mkdir -p /app/downloads /usr/src /JDownloader /content/downloads")
run_cmd("ln -sfn /app /usr/src/app")
run_cmd("chmod -R 777 /app /usr/src/app /JDownloader")

# Ensure Unix line endings on shell scripts
run_cmd("sed -i 's/\r$//' *.sh && chmod +x *.sh")
print("   ✅ Directories and filesystem symlinks mapped (/app -> repo).")

print("🐍 [4/4] Installing Python requirements...")
run_cmd(f"{sys.executable} -m pip install --no-cache-dir sabctools -r requirements.txt")
print("   ✅ Python requirements installed successfully.")

print("\n🎉 Environment setup complete! You are ready to configure the bot.")



In [ ]:
# @title ⚙️ Step 3: Bot Configuration { form-width: "100%" }
# @markdown ### 🔑 1. Required Credentials (Get from @BotFather and my.telegram.org)
BOT_TOKEN = "" # @param {type:"string"}
OWNER_ID = 0 # @param {type:"integer"}
TELEGRAM_API = 0 # @param {type:"integer"}
TELEGRAM_HASH = "" # @param {type:"string"}

# @markdown ---
# @markdown ### 💾 2. Database & Default Storage (Optional)
DATABASE_URL = "" # @param {type:"string"}
DATABASE_NAME = "mltb" # @param {type:"string"}
DEFAULT_UPLOAD = "rc" # @param ["rc", "gd", "bh", "gf"]

# @markdown ---
# @markdown ### 📁 3. Google Drive & Rclone (Optional)
GDRIVE_ID = "" # @param {type:"string"}
IS_TEAM_DRIVE = False # @param {type:"boolean"}
STOP_DUPLICATE = False # @param {type:"boolean"}
INDEX_URL = "" # @param {type:"string"}
RCLONE_PATH = "" # @param {type:"string"}
RCLONE_FLAGS = "" # @param {type:"string"}

# @markdown ---
# @markdown ### ⚡ 4. Telegram & Leech Settings (Optional)
USER_SESSION_STRING = "" # @param {type:"string"}
LEECH_SPLIT_SIZE = 2097152000 # @param {type:"integer"}
AS_DOCUMENT = False # @param {type:"boolean"}
MEDIA_GROUP = False # @param {type:"boolean"}
LEECH_DUMP_CHAT = "" # @param {type:"string"}
AUTHORIZED_CHATS = "" # @param {type:"string"}
SUDO_USERS = "" # @param {type:"string"}
CMD_SUFFIX = "" # @param {type:"string"}

# @markdown ---
# @markdown ### 🌐 5. Web UI & Torrent File Selector (Cloudflare Tunnel)
ENABLE_CLOUDFLARE_TUNNEL = False # @param {type:"boolean"}
SEARCH_API_LINK = "" # @param {type:"string"}
TORRENT_TIMEOUT = 0 # @param {type:"integer"}

# @markdown ---
# @markdown ### 🔄 6. Persistence Options
LOAD_CONFIG_FROM_DRIVE = False # @param {type:"boolean"}
SAVE_CONFIG_TO_DRIVE = True # @param {type:"boolean"}

import os
import re
import sys
import shutil
import subprocess

CONFIG_PATH = "config.py"
DRIVE_CONFIG_PATH = "/content/drive/MyDrive/MLTB/config.py"

# Optional Cloudflare Tunnel setup for BASE_URL
base_url = ""
base_port = 8080

if ENABLE_CLOUDFLARE_TUNNEL:
    print("🌐 Setting up Cloudflare Tunnel for Torrent Web Selection...")
    cf_bin = "/usr/local/bin/cloudflared"
    if not os.path.exists(cf_bin):
        subprocess.run(
            "curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
            shell=True,
            check=True
        )
    # Terminate any old cloudflared
    subprocess.run("pkill -9 -f cloudflared", shell=True)

    # Start cloudflared tunnel in background
    cf_log = "/tmp/cf_tunnel.log"
    subprocess.run(
        f"nohup cloudflared tunnel --url http://localhost:{base_port} --no-autoupdate > {cf_log} 2>&1 &",
        shell=True
    )
    import time
    time.sleep(4)
    # Extract trycloudflare URL from log
    if os.path.exists(cf_log):
        with open(cf_log, "r") as f:
            log_content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
            if match:
                base_url = match.group(0)
                print(f"   ✅ Cloudflare Tunnel Online! Public URL: {base_url}")
            else:
                print("   ⚠️ Could not extract Cloudflare tunnel URL yet. Check /tmp/cf_tunnel.log")

# Check if user wants to load config from Google Drive
config_loaded = False
if LOAD_CONFIG_FROM_DRIVE and os.path.exists(DRIVE_CONFIG_PATH):
    shutil.copy(DRIVE_CONFIG_PATH, CONFIG_PATH)
    print(f"✅ Loaded existing configuration from {DRIVE_CONFIG_PATH}")
    config_loaded = True

if not config_loaded:
    if not BOT_TOKEN or not OWNER_ID or not TELEGRAM_API or not TELEGRAM_HASH:
        raise ValueError("❌ BOT_TOKEN, OWNER_ID, TELEGRAM_API, and TELEGRAM_HASH are mandatory!")

    config_content = f"""# MLTB Configuration (Generated via Google Colab)
# REQUIRED CONFIG
BOT_TOKEN = "{BOT_TOKEN}"
OWNER_ID = {OWNER_ID}
TELEGRAM_API = {TELEGRAM_API}
TELEGRAM_HASH = "{TELEGRAM_HASH}"

# DATABASE CONFIG
DATABASE_URL = "{DATABASE_URL}"
DATABASE_NAME = "{DATABASE_NAME}"

# STORAGE & UPLOAD
DEFAULT_UPLOAD = "{DEFAULT_UPLOAD}"
GDRIVE_ID = "{GDRIVE_ID}"
IS_TEAM_DRIVE = {IS_TEAM_DRIVE}
STOP_DUPLICATE = {STOP_DUPLICATE}
INDEX_URL = "{INDEX_URL}"
RCLONE_PATH = "{RCLONE_PATH}"
RCLONE_FLAGS = "{RCLONE_FLAGS}"
RCLONE_SERVE_URL = ""
RCLONE_SERVE_PORT = 0
RCLONE_SERVE_USER = ""
RCLONE_SERVE_PASS = ""

# TELEGRAM & LEECH
USER_SESSION_STRING = "{USER_SESSION_STRING}"
CMD_SUFFIX = "{CMD_SUFFIX}"
AUTHORIZED_CHATS = "{AUTHORIZED_CHATS}"
SUDO_USERS = "{SUDO_USERS}"
STATUS_LIMIT = 4
STATUS_UPDATE_INTERVAL = 15
LEECH_SPLIT_SIZE = {LEECH_SPLIT_SIZE}
AS_DOCUMENT = {AS_DOCUMENT}
EQUAL_SPLITS = False
MEDIA_GROUP = {MEDIA_GROUP}
USER_TRANSMISSION = False
HYBRID_LEECH = False
LEECH_FILENAME_PREFIX = ""
LEECH_DUMP_CHAT = "{LEECH_DUMP_CHAT}"
CLONE_DUMP_CHATS = ""
THUMBNAIL_LAYOUT = ""
FILES_LINKS = False

# TORRENT / QB / ARIA2 / SABNZBD
TORRENT_TIMEOUT = {TORRENT_TIMEOUT}
BASE_URL = "{base_url}"
BASE_URL_PORT = {base_port if base_url else 0}
WEB_PINCODE = False
EXCLUDED_EXTENSIONS = ""
INCLUDED_EXTENSIONS = ""
INCOMPLETE_TASK_NOTIFIER = False
YT_DLP_OPTIONS = ""
GALLERY_DL_OPTIONS = ""
USE_SERVICE_ACCOUNTS = False
NAME_SUBSTITUTE = r""
FFMPEG_CMDS = {{"merge": ["-f concat -safe 0 -i mltb.txt -c copy mltb.mp4 -del"]}}
UPLOAD_PATHS = {{}}
TG_PROXY = {{}}

# SEARCH ENGINES & PLUGINS
SEARCH_API_LINK = "{SEARCH_API_LINK}"
SEARCH_LIMIT = 0
SEARCH_PLUGINS = [
    "https://raw.githubusercontent.com/qbittorrent/search-plugins/master/nova3/engines/piratebay.py",
    "https://raw.githubusercontent.com/qbittorrent/search-plugins/master/nova3/engines/limetorrents.py",
    "https://raw.githubusercontent.com/qbittorrent/search-plugins/master/nova3/engines/torlock.py",
    "https://raw.githubusercontent.com/qbittorrent/search-plugins/master/nova3/engines/torrentscsv.py",
    "https://raw.githubusercontent.com/qbittorrent/search-plugins/master/nova3/engines/eztv.py",
    "https://raw.githubusercontent.com/qbittorrent/search-plugins/master/nova3/engines/torrentproject.py",
    "https://raw.githubusercontent.com/MaurizioRicci/qBittorrent_search_engines/master/kickass_torrent.py",
    "https://raw.githubusercontent.com/MaurizioRicci/qBittorrent_search_engines/master/yts_am.py",
    "https://raw.githubusercontent.com/MadeOfMagicAndWires/qBit-plugins/master/engines/linuxtracker.py",
    "https://raw.githubusercontent.com/MadeOfMagicAndWires/qBit-plugins/master/engines/nyaasi.py",
    "https://raw.githubusercontent.com/LightDestory/qBittorrent-Search-Plugins/master/src/engines/ettv.py",
    "https://raw.githubusercontent.com/LightDestory/qBittorrent-Search-Plugins/master/src/engines/glotorrents.py",
    "https://raw.githubusercontent.com/LightDestory/qBittorrent-Search-Plugins/master/src/engines/thepiratebay.py",
    "https://raw.githubusercontent.com/v1k45/1337x-qBittorrent-search-plugin/master/leetx.py",
    "https://raw.githubusercontent.com/nindogo/qbtSearchScripts/master/magnetdl.py",
    "https://raw.githubusercontent.com/msagca/qbittorrent_plugins/main/uniondht.py",
    "https://raw.githubusercontent.com/khensolomon/leyts/master/yts.py",
]
USENET_SERVERS = []
JD_EMAIL = ""
JD_PASS = ""
HYDRA_IP = ""
HYDRA_API_KEY = ""
UPSTREAM_REPO = ""
UPSTREAM_BRANCH = "master"
QUEUE_ALL = 0
QUEUE_DOWNLOAD = 0
QUEUE_UPLOAD = 0
RSS_DELAY = 600
RSS_CHAT = ""
RSS_SIZE_LIMIT = 0
"""
    with open(CONFIG_PATH, "w") as f:
        f.write(config_content)
    print(f"✅ Generated {CONFIG_PATH} successfully!")

    # Save to Google Drive if requested
    if SAVE_CONFIG_TO_DRIVE and os.path.exists("/content/drive/MyDrive"):
        os.makedirs("/content/drive/MyDrive/MLTB", exist_ok=True)
        shutil.copy(CONFIG_PATH, DRIVE_CONFIG_PATH)
        print(f"💾 Saved a copy to Google Drive: {DRIVE_CONFIG_PATH}")



In [ ]:
# @title 📂 Step 4: Import Private Files (token.pickle, rclone.conf, accounts) { form-width: "60%" }
# @markdown Link or upload private authentication files:
AUTO_IMPORT_FROM_DRIVE = True # @param {type:"boolean"}
UPLOAD_TOKEN_PICKLE = False # @param {type:"boolean"}
UPLOAD_RCLONE_CONF = False # @param {type:"boolean"}
UPLOAD_ACCOUNTS_ZIP = False # @param {type:"boolean"}
UPLOAD_NETRC = False # @param {type:"boolean"}

import os
import shutil
import zipfile

DRIVE_DIR = "/content/drive/MyDrive/MLTB"

# 1. Auto import from Google Drive
if AUTO_IMPORT_FROM_DRIVE and os.path.exists(DRIVE_DIR):
    for fname in ["token.pickle", "rclone.conf", "accounts.zip", ".netrc"]:
        src = os.path.join(DRIVE_DIR, fname)
        if os.path.exists(src):
            shutil.copy(src, fname)
            print(f"✅ Auto-imported from Drive: {fname}")

# 2. Upload helpers
try:
    from google.colab import files

    if UPLOAD_TOKEN_PICKLE:
        print("📤 Please upload your token.pickle file:")
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn != "token.pickle":
                shutil.move(fn, "token.pickle")
        print("✅ token.pickle saved.")

    if UPLOAD_RCLONE_CONF:
        print("📤 Please upload your rclone.conf file:")
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn != "rclone.conf":
                shutil.move(fn, "rclone.conf")
        print("✅ rclone.conf saved.")

    if UPLOAD_ACCOUNTS_ZIP:
        print("📤 Please upload your accounts.zip file:")
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn != "accounts.zip":
                shutil.move(fn, "accounts.zip")
        print("✅ accounts.zip saved.")

    if UPLOAD_NETRC:
        print("📤 Please upload your .netrc file:")
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn != ".netrc":
                shutil.move(fn, ".netrc")
        print("✅ .netrc saved.")
except Exception as e:
    print(f"ℹ️ File upload skipped or failed: {e}")

# If accounts.zip is present, extract it to accounts/
if os.path.exists("accounts.zip"):
    os.makedirs("accounts", exist_ok=True)
    with zipfile.ZipFile("accounts.zip", "r") as zip_ref:
        zip_ref.extractall("accounts")
    print("✅ accounts.zip unpacked to accounts/ folder.")

# Summary of status
print("\n📊 File Status Checklist:")
print(f"   - token.pickle: {'✅ Present' if os.path.exists('token.pickle') else '❌ Not found'}")
print(f"   - rclone.conf:  {'✅ Present' if os.path.exists('rclone.conf') else '❌ Not found'}")
print(f"   - accounts/:    {'✅ Present' if os.path.isdir('accounts') and os.listdir('accounts') else '❌ Not found'}")
print(f"   - .netrc:       {'✅ Present' if os.path.exists('.netrc') else '❌ Not found (will be auto-created empty)'}")



In [ ]:
# @title 🚀 Step 5: Start Mirror-Leech Bot & View Live Logs { form-width: "40%" }
# @markdown Click Run to start the bot. To stop the bot, click the Stop button on this cell.

import os
import sys
import time
import signal
import subprocess

def cleanup_processes():
    """Kill lingering background daemons before start."""
    pnames = ["aria2c", "qbittorrent-nox", "sabnzbdplus", "gunicorn"]
    for p in pnames:
        subprocess.run(f"pkill -9 -f {p} >/dev/null 2>&1", shell=True)

print("🧹 [1/3] Cleaning up any lingering processes on ports 6800, 8090, 8070...")
cleanup_processes()
time.sleep(1)

print("🔄 [2/3] Running update.py...")
update_res = subprocess.run([sys.executable, "update.py"], text=True, capture_output=True)
if update_res.returncode != 0:
    print(f"⚠️ update.py note:\n{update_res.stderr.strip() or update_res.stdout.strip()}")
else:
    print("   ✅ Config verified by update.py.")

print("🤖 [3/3] Launching Mirror-Leech-Telegram-Bot...")
print("=" * 60)

# Run bot as subprocess to isolate asyncio event loops and stream output live
bot_proc = subprocess.Popen(
    [sys.executable, "-m", "bot"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

try:
    for line in iter(bot_proc.stdout.readline, ""):
        print(line, end="", flush=True)
    bot_proc.wait()
except KeyboardInterrupt:
    print("\n🛑 Stopping bot gracefully (KeyboardInterrupt)...")
    bot_proc.send_signal(signal.SIGINT)
    try:
        bot_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        bot_proc.kill()
    cleanup_processes()
    print("✅ Bot stopped cleanly.")



In [ ]:
# @title 🛑 Emergency Stop & Port Cleanup { form-width: "40%" }
# @markdown Click Run here if you ever need to forcefully stop all bot processes and free up ports.

import subprocess

cmd = (
    "pkill -9 -f aria2c; "
    "pkill -9 -f qbittorrent-nox; "
    "pkill -9 -f sabnzbdplus; "
    "pkill -9 -f gunicorn; "
    "pkill -9 -f cloudflared; "
    "pkill -9 -f 'python3 -m bot'"
)

subprocess.run(cmd, shell=True)
print("✅ All bot processes, daemons, and tunnels terminated. Ports are free.")

